# parte 4 - aprendizagem nao supervisionada
## 4.c analise de associacao (apriori) e 4.d deteccao de outliers (lof)

neste notebook vamos usar o apriori para encontrar regras de associacao
entre as caracteristicas das casas e o lof para identificar outliers.

In [ ]:
# importando as bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor
from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:.2f}'.format
pd.set_option('display.max_rows', 200)
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
# carregando os dados
df = pd.read_csv('train.csv')

## repetindo as transformacoes que o felipe fez

In [ ]:
# target encoding no bairro
from category_encoders import TargetEncoder

encoder = TargetEncoder(cols=['Neighborhood'])
df['Neighborhood'] = encoder.fit_transform(df['Neighborhood'], df['SalePrice'])

In [ ]:
# transformando variaveis categoricas em numeros
colunas_ordinais = ['ExterQual', 'KitchenQual', 'BsmtQual']
mapeamento = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1}

df[colunas_ordinais] = df[colunas_ordinais].replace(mapeamento)

df['BsmtQual'] = df['BsmtQual'].fillna('Po')
mapeamento_completo = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'Po': 0}
df[colunas_ordinais] = df[colunas_ordinais].replace(mapeamento_completo)

In [ ]:
# criando as features novas
df['HouseAge'] = df['YrSold'] - df['YearBuilt']
df['YearsSinceRemodel'] = df['YrSold'] - df['YearRemodAdd']
df['TotalArea'] = df['GrLivArea'] + df['TotalBsmtSF']
df['TotalBath'] = (df['FullBath'] + 0.5*df['HalfBath']
                   + df['BsmtFullBath'] + 0.5*df['BsmtHalfBath'])
df['HasGarage'] = (df['GarageArea'] > 0).astype(int)

In [ ]:
# selecionando as mesmas features
features = [
    'OverallQual', 'TotalArea', 'Neighborhood', 'ExterQual',
    'KitchenQual', 'GarageCars', 'TotalBath', 'BsmtQual',
    'HouseAge', 'YearsSinceRemodel'
]

X = df[features].fillna(0)

# padronizando
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 4.c - analise de associacao com apriori

vamos transformar as features numericas em categorias
para encontrar regras do tipo "se x entao y".

In [ ]:
# transformando as features numericas em categorias
df_apriori = df.copy()

# criando categorias para cada feature
df_apriori['OverallQual_cat'] = pd.cut(df['OverallQual'], bins=3,
    labels=['qualidade_baixa', 'qualidade_media', 'qualidade_alta'])

df_apriori['TotalArea_cat'] = pd.qcut(df['TotalArea'], q=3,
    labels=['area_pequena', 'area_media', 'area_grande'])

df_apriori['GarageCars_cat'] = df['GarageCars'].astype(str).replace(
    {'0': 'sem_garagem', '1': '1_vaga', '2': '2_vagas', '3': '3_vagas', '4': '4+_vagas'})

df_apriori['HouseAge_cat'] = pd.cut(df['HouseAge'], bins=3,
    labels=['casa_nova', 'casa_media', 'casa_antiga'])

df_apriori['SalePrice_cat'] = pd.qcut(df['SalePrice'], q=2,
    labels=['preco_baixo', 'preco_alto'])

df_apriori['TotalBath_cat'] = pd.cut(df['TotalBath'], bins=3,
    labels=['poucos_banheiros', 'banheiros_medio', 'muitos_banheiros'])

df_apriori['ExterQual_cat'] = df['ExterQual'].map(
    {0: 'exterior_pobre', 1: 'exterior_fraco', 2: 'exterior_medio',
     3: 'exterior_bom', 4: 'exterior_excelente'})

# mostrando como ficou
print(df_apriori[['OverallQual_cat', 'TotalArea_cat', 'GarageCars_cat',
                  'HouseAge_cat', 'SalePrice_cat']].head())

In [ ]:
# criando a tabela de transacoes (formato one-hot)
colunas_categoricas = [
    'OverallQual_cat', 'TotalArea_cat', 'GarageCars_cat',
    'HouseAge_cat', 'SalePrice_cat', 'TotalBath_cat', 'ExterQual_cat'
]

transacoes = pd.get_dummies(df_apriori[colunas_categoricas])

print('formato da tabela de transacoes:', transacoes.shape)
print()
print(transacoes.head())